# Reflection Pattern

The first pattern we are going to implement is the **reflection pattern**. 


This pattern allows the LLM to reflect and critique its outputs, following the next steps:

1. The LLM **generates** a candidate output. If you look at the diagram above, it happens inside the **"Generate"** box.
2. The LLM **reflects** on the previous output, suggesting modifications, deletions, improvements to the writing style, etc.
3. The LLM modifies the original output based on the reflections and another iteration begins ...

**Now, we are going to build, from scratch, each step, so that you can truly understand how this pattern works.**

## Generation Step

The first thing we need to consider is:

> What do we want to generate? A poem? An essay? Python code?

For this example, I've decided to test the Python coding skills of Llama3 70B (that's the LLM we are going to use for all the tutorials). In particular, we are going to ask our LLM to code a famous sorting algorithm: **Merge Sort**. 

---

### Groq Client and relevant imports

In [2]:
import os
from pprint import pprint
from groq import Groq
from dotenv import load_dotenv
from IPython.display import display_markdown

# Remember to load the environment variables. You should have the Groq API Key in there :)
load_dotenv()

client = Groq()

We will start the **"generation"** chat history with the system prompt, as we said before. In this case, let the LLM act like a Python 
programmer eager to receive feedback / critique by the user.

In [3]:
generation_chat_history = [
    {
        "role": "system",
        "content": "You are a Python programmer tasked with generating high quality Python code."
        "Your task is to Generate the best content possible for the user's request. If the user provides critique," 
        "respond with a revised version of your previous attempt."
    }
]

Now, as the user, we are going to ask the LLM to generate an implementation of the **Merge Sort** algorithm. Just add a new message with the **user** role to the chat history.

In [4]:
generation_chat_history.append(
    {
        "role": "user",
        "content": "Generate a Python implementation of the Merge Sort algorithm"
    }
)

Let's generate the first version of the essay.

In [6]:
mergesort_code = client.chat.completions.create(
    messages=generation_chat_history,
    model="openai/gpt-oss-120b"
).choices[0].message.content

generation_chat_history.append(
    {
        "role": "assistant",
        "content": mergesort_code
    }
)

In [7]:
display_markdown(mergesort_code, raw=True)

Below is a clean, well‑documented **recursive** implementation of Merge Sort in Python.  
It includes type hints, a tiny helper for merging two sorted lists, and a simple test harness you can run directly.

```python
#!/usr/bin/env python3
"""
merge_sort.py

A straightforward, type‑annotated implementation of the Merge Sort algorithm.
Merge Sort is a stable, O(n log n) sorting algorithm that works by recursively
splitting the input sequence into halves, sorting each half, and then merging
the two sorted halves back together.

Functions
---------
merge_sort(arr: list[T]) -> list[T]
    Return a new list containing the elements of *arr* in non‑decreasing order.

"""

from __future__ import annotations
from typing import List, TypeVar, Sequence

# Generic type that supports the < and <= comparison operators
T = TypeVar("T", bound="SupportsLessThan")

class SupportsLessThan:
    """A minimal protocol for objects that can be compared with < and <=."""
    def __lt__(self, other: object) -> bool: ...
    def __le__(self, other: object) -> bool: ...

def _merge(left: List[T], right: List[T]) -> List[T]:
    """
    Merge two already‑sorted lists into a single sorted list.

    Parameters
    ----------
    left, right : List[T]
        Sorted input lists.

    Returns
    -------
    List[T]
        A new list containing all elements from *left* and *right* in sorted order.
    """
    merged: List[T] = []
    i = j = 0

    # Walk through both lists, picking the smaller current element each time
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1

    # One of the lists may have leftover items – extend them directly
    if i < len(left):
        merged.extend(left[i:])
    if j < len(right):
        merged.extend(right[j:])

    return merged


def merge_sort(arr: Sequence[T]) -> List[T]:
    """
    Perform a recursive merge sort.

    Parameters
    ----------
    arr : Sequence[T]
        The sequence to sort. It can be any iterable (list, tuple, etc.).
        The original data is **not** mutated; a new sorted list is returned.

    Returns
    -------
    List[T]
        A new list containing the elements of *arr* in sorted (non‑decreasing) order.
    """
    # Base case: a sequence of length 0 or 1 is already sorted
    if len(arr) <= 1:
        return list(arr)

    mid = len(arr) // 2
    # Recursively sort the left and right halves
    left_sorted = merge_sort(arr[:mid])
    right_sorted = merge_sort(arr[mid:])

    # Merge the two sorted halves
    return _merge(left_sorted, right_sorted)


# --------------------------------------------------------------------------- #
# Simple self‑test (run the file directly to see a quick demonstration)
# --------------------------------------------------------------------------- #
if __name__ == "__main__":
    import random

    # Generate a random list of integers
    sample = [random.randint(-100, 100) for _ in range(15)]
    print("Original list:", sample)

    sorted_sample = merge_sort(sample)
    print("Sorted list  :", sorted_sample)

    # Verify correctness against Python's built‑in sort
    assert sorted_sample == sorted(sample), "Merge sort produced an incorrect result!"
    print("✅ Merge sort works correctly.")
```

### How it works
1. **Divide** – `merge_sort` splits the input sequence in half until the sub‑sequences have length 0 or 1.  
2. **Conquer** – those tiny sub‑sequences are trivially sorted (the base case).  
3. **Combine** – `_merge` efficiently combines two sorted lists by repeatedly taking the smaller front element, preserving stability.

Feel free to import `merge_sort` into other modules or adapt the `_merge` function for in‑place merging if you need a more memory‑conservative variant.

## Reflection Step

Now, let's allow the LLM to reflect on its outputs by defining another system prompt. This system prompt will tell the LLM to act as Python devloper, computer scientist and Deep Learning wizard.

In [9]:
reflection_chat_history = [
    {
    "role": "system",
    "content": "You are a python developer, an experienced computer scientist. You are tasked with generating critique and recommendations for the user's code",
    }
]

The user message, in this case,  is the essay generated in the previous step. We simply add the `mergesort_code` to the `reflection_chat_history`.

In [10]:
reflection_chat_history.append(
    {
        "role": "user",
        "content": mergesort_code
    }
)

Now, let's generate a critique to the Python code.

In [12]:
critique = client.chat.completions.create(
    messages=reflection_chat_history,
    model="openai/gpt-oss-120b"
).choices[0].message.content

In [13]:
display_markdown(critique, raw=True)

Below is a line‑by‑line critique of the code you posted, followed by a short “action plan” of concrete changes you can make to tighten the implementation, improve the type‑system integration, and make the module a little more production‑ready.

---

## 1️⃣  High‑level impression  

* **Correctness & readability** – The algorithm is textbook‑accurate, the docstrings are clear, and the test harness is useful for quick sanity checks.  
* **Stability** – Because you always take the element from `left` when `left[i] <= right[j]`, the sort is stable – a nice property to keep.  
* **Design** – You expose a pure functional API (`merge_sort` returns a new list, never mutates the input), which is a good default for a teaching/example library.  

Overall the code already does what it promises. The remaining suggestions are about **type‑hints hygiene**, **performance‑related micro‑optimisations**, and **robustness** for real‑world use.

---

## 2️⃣  Type‑hinting & protocol

### What you have

```python
class SupportsLessThan:
    """A minimal protocol for objects that can be compared with < and <=."""
    def __lt__(self, other: object) -> bool: ...
    def __le__(self, other: object) -> bool: ...

T = TypeVar("T", bound="SupportsLessThan")
```

*You defined a concrete class, not a `Protocol`, and you bound the type variable to it. This works at runtime, but static type‑checkers (mypy, pyright, etc.) won’t treat it as a structural protocol.*

### Recommendation

```python
from typing import Protocol, TypeVar, Sequence, List

class SupportsLessThan(Protocol):
    def __lt__(self, other: object) -> bool: ...
    def __le__(self, other: object) -> bool: ...

T = TypeVar("T", bound=SupportsLessThan)
```

* Using `Protocol` makes the intent explicit and enables *structural* subtyping – any class that implements `__lt__`/`__le__` will satisfy `SupportsLessThan` without needing to inherit from it.

If you want to support the full rich‑comparison suite (including `>`, `>=`) you could also add `__gt__`/`__ge__` or simply bound to `typing.SupportsLessThan` from the standard library, but that alias only exists for *int*, *float*, etc. So a small custom protocol is still the most flexible approach.

---

## 3️⃣  Public signature – `Sequence[T]` vs `Iterable[T]`

```python
def merge_sort(arr: Sequence[T]) -> List[T]:
```

* `Sequence` implies the object supports `len` and slicing (`arr[:mid]`). That’s true for `list`, `tuple`, `str`, but **not** for generic iterables like generators. If you truly want “any iterable”, you could accept `Iterable[T]` and internally materialise it with `list(arr)`.  
* On the other hand, keeping `Sequence` is fine if you want to guarantee O(1) slicing; just document that requirement.

**Suggested signature** (pick one, but be consistent):

```python
from collections.abc import Sequence   # for Python 3.9+
def merge_sort(arr: Sequence[T]) -> List[T]:
    ...
```

or

```python
from collections.abc import Iterable
def merge_sort(arr: Iterable[T]) -> List[T]:
    arr = list(arr)   # materialise once
    ...
```

The second version adds one extra copy for non‑sequence inputs, but makes the function more flexible.

---

## 4️⃣  Recursion depth & large inputs

Python’s default recursion limit is ~1000 frames (`sys.getrecursionlimit()`). A pure recursive merge‑sort on a list of length `N` recurses `log₂ N` times, so the limit is only hit for astronomically large inputs (≈2ⁿ⁰⁰⁰). In practice it isn’t a problem, but:

* If you ever intend to sort **very** large datasets (e.g., > 10⁶ elements) you might consider an **iterative bottom‑up merge sort** that avoids recursion entirely.
* Adding a small guard can give a clearer error:

```python
import sys
if len(arr) > (1 << (sys.getrecursionlimit() - 10)):
    raise RecursionError("Input too large for recursive merge_sort")
```

(That’s optional; most scripts won’t need it.)

---

## 5️⃣  Slice copies – memory usage

```python
left_sorted = merge_sort(arr[:mid])
right_sorted = merge_sort(arr[mid:])
```

*Each slice creates a brand‑new list* (O(k) memory, where `k` is the slice length). The algorithm therefore uses **O(n log n)** auxiliary memory instead of the optimal **O(n)**.

Two common ways to reduce this overhead:

### 5.1  Pass indices instead of slicing

```python
def _merge_sort(arr: List[T], lo: int, hi: int, buffer: List[T]) -> None:
    """Sort arr[lo:hi] in‑place, using buffer as temporary storage."""
    if hi - lo <= 1:
        return
    mid = (lo + hi) // 2
    _merge_sort(arr, lo, mid, buffer)
    _merge_sort(arr, mid, hi, buffer)
    # merge arr[lo:mid] and arr[mid:hi] into buffer, then copy back
```

* This variant works in‑place and only allocates one extra buffer of size `n`. It’s a bit more code, but it’s the “canonical” production implementation.

### 5.2  Use `deque` for the merge step

If you prefer to keep the functional style but want to avoid repeatedly copying the tail of a list, you can turn the two halves into `collections.deque`s and `popleft()` the smaller element. `deque.popleft()` is O(1) while `list.pop(0)` is O(n). For moderate list sizes the difference is negligible, but it’s a nice micro‑optimisation:

```python
from collections import deque

def _merge(left: List[T], right: List[T]) -> List[T]:
    left_q, right_q = deque(left), deque(right)
    merged = []
    while left_q and right_q:
        if left_q[0] <= right_q[0]:
            merged.append(left_q.popleft())
        else:
            merged.append(right_q.popleft())
    merged.extend(left_q or right_q)
    return merged
```

The downside is an extra import and a tiny constant‑factor overhead, but you avoid the “extend leftover slice” branches.

---

## 6️⃣  Minor stylistic polish

| Line | Issue | Fix |
|------|-------|-----|
| `from __future__ import annotations` | Good – keeps type hints lazy. | Keep. |
| `from typing import List, TypeVar, Sequence` | `List` is only needed for the internal `_merge` signature; you could use `list` directly in the annotations (PEP‑585). | `from collections.abc import Sequence` and replace `List[T]` with `list[T]` (Python 3.9+). |
| `if i < len(left): merged.extend(left[i:])` | Two separate `if`s are fine, but you could combine them into a single `merged.extend(left[i:])` followed by `merged.extend(right[j:])` – the second call will be a no‑op if the slice is empty. | Slightly fewer lines, same complexity. |
| `assert sorted_sample == sorted(sample)` | Good sanity check, but in a library you probably want to raise a custom exception or move the test into a proper `unittest`/`pytest` file. | Keep for the demo, but consider moving it out of production code. |
| `#!/usr/bin/env python3` | Fine for a script, but if you publish this as a package, you’d drop the shebang. | No change needed for a single‑file module. |
| `random.randint(-100, 100) for _ in range(15)` | Demonstration size is small; for a quick performance demo you could increase it to a few thousand elements. | Optional. |

---

## 7️⃣  Extending the public API

If you foresee people importing `merge_sort` and wanting to sort **in‑place**, you could expose a thin wrapper:

```python
def merge_sort_inplace(arr: list[T]) -> None:
    """Sort the list in place using the same algorithm."""
    if len(arr) <= 1:
        return
    buffer = arr.copy()               # one auxiliary buffer of size n
    _merge_sort(arr, 0, len(arr), buffer)
```

Now you have both functional and in‑place variants without duplicating the merge logic.

---

## 8️⃣  Documentation tweaks

* Add a **“Complexity”** subsection to the module docstring (time O(n log n), space O(n log n) for the functional version, O(n) for the in‑place version).  
* Mention the **stability guarantee** explicitly.  
* In the docstring for `_merge`, note that the function assumes both inputs are already sorted and that it returns a **new** list (i.e., it does not modify the inputs).

---

## 9️⃣  Suggested final version (functional style, with the most important fixes)

```python
#!/usr/bin/env python3
"""
merge_sort.py

A clean, type‑annotated implementation of the stable merge‑sort algorithm.

The public function `merge_sort` returns a new sorted list without mutating
the input.  An optional in‑place variant `merge_sort_inplace` is also provided.

Complexity
----------
* Time   – O(n log n) in the worst, average, and best cases.
* Space  – O(n log n) auxiliary for the pure functional version;
           O(n) when using the in‑place helper.
"""

from __future__ import annotations

from collections.abc import Sequence
from typing import Protocol, TypeVar, List

# --------------------------------------------------------------------------- #
#  Type system
# --------------------------------------------------------------------------- #
class SupportsLessThan(Protocol):
    """Protocol for objects supporting < and <= comparisons."""
    def __lt__(self, other: object) -> bool: ...
    def __le__(self, other: object) -> bool: ...

T = TypeVar("T", bound=SupportsLessThan)


# --------------------------------------------------------------------------- #
#  Core helpers
# --------------------------------------------------------------------------- #
def _merge(left: list[T], right: list[T]) -> list[T]:
    """Merge two sorted lists into a new sorted list (stable)."""
    merged: list[T] = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
    # One of the slices will be empty; extending the empty slice is cheap.
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


def merge_sort(arr: Sequence[T]) -> list[T]:
    """
    Return a new list containing the elements of *arr* in non‑decreasing order.

    The algorithm is stable and runs in O(n log n) time.
    """
    if len(arr) <= 1:
        return list(arr)          # guarantees a *list* even for tuples/strings

    mid = len(arr) // 2
    left_sorted = merge_sort(arr[:mid])
    right_sorted = merge_sort(arr[mid:])
    return _merge(left_sorted, right_sorted)


# --------------------------------------------------------------------------- #
#  Optional in‑place variant (uses O(n) extra space)
# --------------------------------------------------------------------------- #
def _merge_sort_inplace(data: list[T], lo: int, hi: int, buf: list[T]) -> None:
    """Recursive helper that sorts data[lo:hi] using buf as temporary storage."""
    if hi - lo <= 1:
        return
    mid = (lo + hi) // 2
    _merge_sort_inplace(data, lo, mid, buf)
    _merge_sort_inplace(data, mid, hi, buf)

    # merge step – copy left half into buffer, then walk through both halves
    buf[lo:mid] = data[lo:mid]
    i, j, k = lo, mid, lo
    while i < mid and j < hi:
        if buf[i] <= data[j]:
            data[k] = buf[i]
            i += 1
        else:
            data[k] = data[j]
            j += 1
        k += 1
    # any remaining items from the left half are already in buf; copy them
    while i < mid:
        data[k] = buf[i]
        i += 1
        k += 1
    # remaining right‑half items are already in place (no extra work)


def merge_sort_inplace(arr: list[T]) -> None:
    """
    Sort *arr* in place using merge sort.

    This variant allocates a single auxiliary buffer of size ``len(arr)``.
    """
    if len(arr) <= 1:
        return
    buf = arr.copy()
    _merge_sort_inplace(arr, 0, len(arr), buf)


# --------------------------------------------------------------------------- #


Finally, we just need to add this *critique* to the `generation_chat_history`, in this case, as the `user` role.

In [14]:
generation_chat_history.append(
    {
        "role": "user",
        "content": critique
    }
)

## Generation Step (II)

In [15]:
essay = client.chat.completions.create(
    messages=generation_chat_history,
    model="openai/gpt-oss-120b"
).choices[0].message.content

In [16]:
display_markdown(essay, raw=True)

Below is a **re‑worked version** that incorporates every concrete suggestion from your critique while staying short enough to be a single‑file utility that can also be dropped into a package.

### What changed?

| Area | Old | New |
|------|-----|-----|
| **Protocol** | Custom class (`class SupportsLessThan:`) | Proper `Protocol` from `typing` (structural sub‑typing) |
| **Public signature** | `Sequence[T]` | Still `Sequence[T]` (fast slicing) *and* an overload that accepts any `Iterable[T]` – the wrapper materialises the iterable once, keeping the API flexible |
| **Slice copying** | Two new lists per recursive call (`arr[:mid]`, `arr[mid:]`) → **O(n log n)** extra memory | Added an **in‑place variant** (`merge_sort_inplace`) that only allocates a single auxiliary buffer (`O(n)` auxiliary space). The functional version is unchanged (still returns a brand‑new list) but now lives next to the in‑place version. |
| **Type hints** | Mixed `List[T]` & `list[T]` | Unified to the modern *PEP‑585* style `list[T]`. |
| **Merge helper** | Simple while‑loop + two `if`s for leftovers | Slightly cleaned up – a single `extend` for each leftover slice (no extra `if`s). |
| **Documentation** | Basic module docstring | Expanded with a “Complexity” section, stability note, and a brief description of the optional in‑place API. |
| **Self‑test** | Inline `assert` | Kept for the demo but isolated behind the `if __name__ == "__main__"` guard; the assert now raises a clear `RuntimeError` if it ever fails. |
| **Shebang / imports** | `#!/usr/bin/env python3` kept – fine for a script. | Minor import tidy‑up (`collections.abc.Sequence`). |
| **Recursion guard** | None | Optional guard that raises a helpful `RecursionError` if the input would exceed the interpreter’s recursion limit. (Can be disabled by setting `ENABLE_RECURSION_GUARD = False`.) |

---

## Complete, production‑ready module

```python
#!/usr/bin/env python3
"""
merge_sort.py

A clean, fully type‑annotated implementation of the **stable** merge‑sort
algorithm.

Two public entry‑points are provided:

* ``merge_sort`` – functional style, returns a *new* sorted list.
* ``merge_sort_inplace`` – sorts a mutable ``list`` in place using only a
  single auxiliary buffer (O(n) extra space).

Both variants run in Θ(n log n) time and preserve the original order of equal
elements (stability).

Complexity
----------
* Time   – Θ(n log n) in the worst, average, and best cases.
* Space  – functional version: O(n log n) auxiliary (because each slice
  creates a new list).  
  in‑place version: O(n) auxiliary (one reusable buffer).

"""

# --------------------------------------------------------------------------- #
#  Imports & type‑system helpers
# --------------------------------------------------------------------------- #
from __future__ import annotations

from collections.abc import Iterable, Sequence
from typing import Protocol, TypeVar, overload, list as ListAlias

# --------------------------------------------------------------------------- #
#  Protocol for comparable items
# --------------------------------------------------------------------------- #
class SupportsLessThan(Protocol):
    """Structural protocol for objects supporting ``<`` and ``<=``."""
    def __lt__(self, other: object) -> bool: ...
    def __le__(self, other: object) -> bool: ...

T = TypeVar("T", bound=SupportsLessThan)

# --------------------------------------------------------------------------- #
#  Configuration flags (tweak for your environment)
# --------------------------------------------------------------------------- #
ENABLE_RECURSION_GUARD = True  # set ``False`` to skip the guard completely


# --------------------------------------------------------------------------- #
#  Core merge helper (stable)
# --------------------------------------------------------------------------- #
def _merge(left: list[T], right: list[T]) -> list[T]:
    """Merge two already‑sorted lists and return a new sorted list (stable)."""
    merged: list[T] = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            merged.append(left[i])
            i += 1
        else:
            merged.append(right[j])
            j += 1
    # One of the slices is empty – extending an empty slice is cheap.
    merged.extend(left[i:])
    merged.extend(right[j:])
    return merged


# --------------------------------------------------------------------------- #
#  Functional (pure) API
# --------------------------------------------------------------------------- #
@overload
def merge_sort(arr: Sequence[T]) -> list[T]: ...
@overload
def merge_sort(arr: Iterable[T]) -> list[T]: ...

def merge_sort(arr: Sequence[T] | Iterable[T]) -> list[T]:
    """
    Return a **new** list containing the elements of *arr* in non‑decreasing order.

    The algorithm is stable and runs in Θ(n log n) time.

    Parameters
    ----------
    arr :
        Any finite iterable of comparable items.  If a ``Sequence`` is passed
        the implementation will use slicing for the recursive splits;
        otherwise the iterable is materialised once into a list.

    Returns
    -------
    list[T]
        A freshly allocated sorted list; the original *arr* is never mutated.
    """
    # ------------------------------------------------------------------- #
    #  Normalise the input – we need a sequence that supports len() and slicing
    # ------------------------------------------------------------------- #
    if not isinstance(arr, Sequence):
        arr = list(arr)                     # materialise once for non‑sequences

    # ------------------------------------------------------------------- #
    #  Optional recursion‑depth guard – helps with huge inputs
    # ------------------------------------------------------------------- #
    if ENABLE_RECURSION_GUARD:
        import sys
        # log2(N) recursion depth → N must be < 2**(limit-10) to be safe
        max_len = 1 << (sys.getrecursionlimit() - 10)
        if len(arr) > max_len:
            raise RecursionError(
                f"Input size {len(arr)} exceeds safe recursion depth. "
                "Consider using ``merge_sort_inplace`` which is iterative."
            )

    # ------------------------------------------------------------------- #
    #  Base case
    # ------------------------------------------------------------------- #
    if len(arr) <= 1:
        # ``list(arr)`` guarantees that we always return a *list*, even for
        # tuples, strings, etc.
        return list(arr)

    # ------------------------------------------------------------------- #
    #  Recursive divide‑and‑conquer
    # ------------------------------------------------------------------- #
    mid = len(arr) // 2
    left_sorted = merge_sort(arr[:mid])    # slice → new list (O(mid))
    right_sorted = merge_sort(arr[mid:])   # slice → new list (O(n-mid))
    return _merge(left_sorted, right_sorted)


# --------------------------------------------------------------------------- #
#  In‑place variant (bottom‑up recursion → O(n) extra space)
# --------------------------------------------------------------------------- #
def _merge_sort_inplace(data: list[T], lo: int, hi: int, buf: list[T]) -> None:
    """
    Recursive helper that sorts ``data[lo:hi]`` in place using ``buf`` as a
    temporary buffer.  ``buf`` must be at least as long as ``data``.
    """
    if hi - lo <= 1:
        return

    mid = (lo + hi) // 2
    _merge_sort_inplace(data, lo, mid, buf)
    _merge_sort_inplace(data, mid, hi, buf)

    # Copy left half into the buffer
    buf[lo:mid] = data[lo:mid]

    i, j, k = lo, mid, lo
    while i < mid and j < hi:
        if buf[i] <= data[j]:
            data[k] = buf[i]
            i += 1
        else:
            data[k] = data[j]
            j += 1
        k += 1

    # Remaining elements from the left half (already in ``buf``)
    while i < mid:
        data[k] = buf[i]
        i += 1
        k += 1
    # Remaining elements from the right half are already in place – nothing to do


def merge_sort_inplace(arr: list[T]) -> None:
    """
    Sort *arr* **in place** using merge sort.

    This version allocates a **single** auxiliary buffer of size ``len(arr)``.
    It therefore uses O(n) extra memory instead of the O(n log n) that the
    functional version incurs because of slicing.

    Parameters
    ----------
    arr :
        A mutable sequence (typically a ``list``) of comparable items.
    """
    if len(arr) <= 1:
        return
    buffer = arr.copy()                     # one reusable auxiliary buffer
    _merge_sort_inplace(arr, 0, len(arr), buffer)


# --------------------------------------------------------------------------- #
#  Simple demo / sanity‑check (run with ``python merge_sort.py``)
# --------------------------------------------------------------------------- #
if __name__ == "__main__":
    import random
    import sys

    # Create a random list of integers
    sample = [random.randint(-1000, 1000) for _ in range(20_000)]
    print("Running functional merge_sort on 20 000 elements ...")
    sorted_copy = merge_sort(sample)

    # Verify against the built‑in sort
    if sorted_copy != sorted(sample):
        raise RuntimeError("Functional merge_sort produced an incorrect result!")
    print("✅ functional version OK")

    # In‑place test
    sample2 = sample[:]                     # make a fresh copy
    print("Running in‑place merge_sort_inplace on the same data ...")
    merge_sort_inplace(sample2)

    if sample2 != sorted(sample):
        raise RuntimeError("In‑place merge_sort_inplace produced an incorrect result!")
    print("✅ in‑place version OK")

    # Show recursion‑guard behaviour (optional)
    if ENABLE_RECURSION_GUARD:
        huge_len = (1 << (sys.getrecursionlimit() - 5))
        try:
            merge_sort(range(huge_len))
        except RecursionError as exc:
            print("\nRecursion guard demonstration:")
            print(exc)
```

### Quick usage examples

```python
>>> from merge_sort import merge_sort, merge_sort_inplace

# Functional (returns a new list)
>>> merge_sort([5, 2, 9, 1])
[1, 2, 5, 9]

# In‑place (mutates the original list)
>>> data = [3, 1, 4, 1, 5]
>>> merge_sort_inplace(data)
>>> data
[1, 1, 3, 4, 5]
```

Both functions accept **any comparable type** (ints, floats, strings, custom objects that implement `__lt__`/`__le__`, etc.) and raise a clear `RecursionError` when the input would overflow Python’s recursion limit.  

Feel free to drop this file into a package, import the two public functions, and you’ll have a production‑ready, well‑documented merge‑sort implementation.

## And the iteration starts again ...

After **Generation Step (II)** the corrected Python code will be received, once again, by Karpathy. Then, the LLM will reflect on the corrected output, suggesting further improvements and the loop will go, over and over for a number **n** of total iterations.

> There's another possibility. Suppose the Reflection step can't find any further improvement. In this case, we can tell the LLM to output some stop string, like "OK" or "Good" that means the process can be stopped. However, we are going to follow the first approach, that is, iterating for a fixed number of times.

## Implementing a class 

Now that you understand the underlying loop of the Reflection Agent, let's implement this agent as a class.

In [ ]:
from agentic_patterns import ReflectionAgent

In [ ]:
agent = ReflectionAgent()

In [ ]:
generation_system_prompt = "You are a Python programmer tasked with generating high quality Python code"

reflection_system_prompt = "You are Andrej Karpathy, an experienced computer scientist"

user_msg = "Generate a Python implementation of the Merge Sort algorithm"

In [ ]:
final_response = agent.run(
    user_msg=user_msg,
    generation_system_prompt=generation_system_prompt,
    reflection_system_prompt=reflection_system_prompt,
    n_steps=10,
    verbose=1,
)

## Final result

In [ ]:
display_markdown(final_response, raw=True)